## Re-split dataset(Garbage detection-6 categories).

######Due to the imbalanced class distribution across the available splits (train, valid, test), re-splitting the data into balanced sets is mandatory.

In [1]:
import kagglehub, yaml, shutil
import numpy as np, pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# 1) Download (cached)
dl = Path(kagglehub.dataset_download('viswaprakash1990/garbage-detection'))
SRC = sorted(dl.rglob('data.yaml'))[0].parent

# 2) Class names from the original file
names = yaml.safe_load(open(SRC / 'data.yaml'))['names']
NAMES = {int(k): str(v) for k, v in names.items()} if isinstance(names, dict) else dict(enumerate(names))
print('Classes:', NAMES)

# 3) Clean folder: images symlinked, labels copied fresh, zero-area boxes removed
DATASET_ROOT = Path('/content/garbage')
DATASET_ROOT.mkdir(exist_ok=True)
removed = 0
for split in ['train', 'valid', 'test']:
    d = DATASET_ROOT / split
    if d.is_symlink(): d.unlink()
    d.mkdir(exist_ok=True)
    img = d / 'images'
    if not img.exists(): img.symlink_to(SRC / split / 'images', target_is_directory=True)
    lbl = d / 'labels'
    if lbl.exists(): shutil.rmtree(lbl)
    shutil.copytree(SRC / split / 'labels', lbl, copy_function=shutil.copyfile)
    for f in lbl.glob('*.txt'):
        lines = f.read_text().splitlines()
        keep = [l for l in lines if len(l.split()) == 5 and float(l.split()[3]) > 0 and float(l.split()[4]) > 0]
        removed += sum(1 for l in lines if l.strip()) - len(keep)
        f.write_text('\n'.join(keep) + ('\n' if keep else ''))
print('Zero-area lines removed:', removed, '(expected 17)')

# 4) data.yaml + output folder
cfg = {'path': str(DATASET_ROOT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
       'nc': len(names), 'names': names}
DATA_YAML = DATASET_ROOT / 'data.yaml'
yaml.safe_dump(cfg, open(DATA_YAML, 'w'), sort_keys=False)
OUTPUT = Path('/content/drive/MyDrive/garbage_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)

for s in ['train', 'valid', 'test']:
    print(s, len(list((DATASET_ROOT / s / 'images').iterdir())), 'images')

Mounted at /content/drive
Using Colab cache for faster access to the 'garbage-detection' dataset.
Classes: {0: 'BIODEGRADABLE', 1: 'CARDBOARD', 2: 'GLASS', 3: 'METAL', 4: 'PAPER', 5: 'PLASTIC'}
Zero-area lines removed: 17 (expected 17)
train 7324 images
valid 2098 images
test 1042 images


In [ ]:
!pip -q install iterative-stratification
import numpy as np, shutil
from scipy.stats import ks_2samp
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

C = [NAMES[k] for k in sorted(NAMES)]
src_group = lambda stem: stem.split('.rf.')[0]   # Roboflow source image id

def load_split(split):
    rows = []
    for img in sorted((DATASET_ROOT / split / 'images').iterdir()):
        lbl = DATASET_ROOT / split / 'labels' / (img.stem + '.txt')
        cls = [int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
        r = {'orig_split': split, 'image': img.name, 'stem': img.stem,
             'group': src_group(img.stem), 'n_obj': len(cls)}
        r.update({NAMES[k]: cls.count(k) for k in sorted(NAMES)})
        rows.append(r)
    return pd.DataFrame(rows)

meta = pd.concat([load_split(s) for s in ['train', 'valid', 'test']], ignore_index=True)
pool = meta[meta.orig_split.isin(['valid', 'test'])].copy()

# Leakage already present with train (same in both versions; train is not changed)
print('Pool groups also in train:', len(set(pool.group) & set(meta[meta.orig_split == 'train'].group)))

# Group-level multi-label stratification: class presence + density bin
g = pool.groupby('group').agg(n_obj=('n_obj', 'sum'), **{c: (c, 'sum') for c in C})
dbin = pd.cut(g.n_obj, [-1, 1, 5, 20, np.inf], labels=False).astype(int)
Y = np.hstack([(g[C].values > 0).astype(int), np.eye(4, dtype=int)[dbin]])
va_idx, te_idx = next(MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
                      .split(np.zeros(len(g)), Y))
assign = pd.Series('valid', index=g.index); assign.iloc[te_idx] = 'test'
pool['new_split'] = pool.group.map(assign)

def split_report(df, col):
    v, t = df[df[col] == 'valid'], df[df[col] == 'test']
    bv, bt = v[C].sum(), t[C].sum()
    return {'valid images': len(v), 'test images': len(t),
            'min test boxes': f'{int(bt.min())} ({bt.idxmin()})',
            'class-share TV dist': round(0.5 * np.abs(bv / bv.sum() - bt / bt.sum()).sum(), 4),
            'density KS': round(ks_2samp(v.n_obj, t.n_obj).statistic, 4),
            'valid mean obj/img': round(v.n_obj.mean(), 2), 'test mean obj/img': round(t.n_obj.mean(), 2),
            'shared groups valid-test': len(set(v.group) & set(t.group))}

A, B = split_report(pool, 'orig_split'), split_report(pool, 'new_split')
display(pd.DataFrame({'official': A, 're-split': B}))

# Per-class boxes in each version
per_cls = pd.concat({'official': pool.groupby('orig_split')[C].sum().T,
                     're-split': pool.groupby('new_split')[C].sum().T}, axis=1)
display(per_cls)

adopt = (int(B['min test boxes'].split()[0]) > 0 and B['class-share TV dist'] < A['class-share TV dist']
         and B['density KS'] < A['density KS'] and B['shared groups valid-test'] == 0)
print('DECISION:', 'ADOPT re-split' if adopt else 'KEEP official split')

Pool groups also in train: 0


,official,re-split
valid images,2098,1571
test images,1042,1569
min test boxes,0 (GLASS),638 (CARDBOARD)
class-share TV dist,0.8956,0.0382
density KS,0.1669,0.0144
valid mean obj/img,9.01,7.36
test mean obj/img,3.42,6.95
shared groups valid-test,0,0


official        re-split      
orig_split        test  valid     test valid
BIODEGRADABLE       49  13632     6690  6991
CARDBOARD           20   1291      638   673
GLASS                0   2380     1045  1335
METAL              533   1360      815  1078
PAPER             1375     33      752   656
PLASTIC           1585    214      971   828

DECISION: ADOPT re-split


In [ ]:
assert adopt, 'Decision was KEEP — do not build the re-split.'
NEW_ROOT = Path('/content/garbage_resplit')
for s in ['valid', 'test']:
    for sub in ['images', 'labels']:
        d = NEW_ROOT / s / sub
        if d.exists(): shutil.rmtree(d)
        d.mkdir(parents=True)
tr = NEW_ROOT / 'train'
if tr.is_symlink(): tr.unlink()
tr.symlink_to(DATASET_ROOT / 'train', target_is_directory=True)   # train unchanged

for r in pool.itertuples():
    (NEW_ROOT / r.new_split / 'images' / r.image).symlink_to((DATASET_ROOT / r.orig_split / 'images' / r.image).resolve())
    lbl = DATASET_ROOT / r.orig_split / 'labels' / (r.stem + '.txt')   # cleaned labels
    if lbl.exists():
        (NEW_ROOT / r.new_split / 'labels' / (r.stem + '.txt')).symlink_to(lbl)

cfg_new = {'path': str(NEW_ROOT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
           'nc': len(names), 'names': names}
yaml.safe_dump(cfg_new, open(NEW_ROOT / 'data.yaml', 'w'), sort_keys=False)
pool[['image', 'group', 'orig_split', 'new_split']].to_csv(OUTPUT / 'resplit_assignment.csv', index=False)

for s in ['train', 'valid', 'test']:
    print(s, len(list((NEW_ROOT / s / 'images').iterdir())), 'images')

train 7324 images
valid 1571 images
test 1569 images


In [ ]:
final = meta.copy()
final['split'] = final['orig_split']
final.loc[pool.index, 'split'] = pool['new_split']
S = ['train', 'valid', 'test']

# Images per class (an image with several classes counts once in each)
imgs = pd.DataFrame({s: (final[final.split == s][C] > 0).sum() for s in S})
imgs.loc['All images (unique)'] = [int((final.split == s).sum()) for s in S]
imgs['total'] = imgs.sum(axis=1)

# Boxes per class
boxes = pd.DataFrame({s: final[final.split == s][C].sum() for s in S})
boxes.loc['Total boxes'] = boxes.sum()
boxes['total'] = boxes.sum(axis=1)

# Share of each class's boxes going to each split (%)
pct = (boxes.drop('Total boxes')[S].div(boxes.drop('Total boxes')['total'], axis=0) * 100).round(1)

print('Images per class:'); display(imgs.astype(int))
print('Boxes per class:'); display(boxes.astype(int))
print('Split share of each class boxes (%):'); display(pct)

final[['image', 'group', 'split'] + C + ['n_obj']].to_csv(OUTPUT / 'final_split_stats.csv', index=False)

Images per class:


,train,valid,test,total
BIODEGRADABLE,1603,342,342,2287
CARDBOARD,1053,224,225,1502
GLASS,1879,408,397,2684
METAL,1328,285,285,1898
PAPER,1238,265,265,1768
PLASTIC,1019,222,223,1464
All images (unique),7324,1571,1569,10464


Boxes per class:


,train,valid,test,total
BIODEGRADABLE,31714,6991,6690,45395
CARDBOARD,3385,673,638,4696
GLASS,5429,1335,1045,7809
METAL,3948,1078,815,5841
PAPER,2979,656,752,4387
PLASTIC,4146,828,971,5945
Total boxes,51601,11561,10911,74073


Split share of each class boxes (%):


,train,valid,test
BIODEGRADABLE,69.9,15.4,14.7
CARDBOARD,72.1,14.3,13.6
GLASS,69.5,17.1,13.4
METAL,67.6,18.5,14.0
PAPER,67.9,15.0,17.1
PLASTIC,69.7,13.9,16.3


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = {'train': DATASET_ROOT / 'train' / 'images',
          'val':   DATASET_ROOT / 'valid' / 'images',
          'test':  DATASET_ROOT / 'test'  / 'images'}
for k, v in SPLITS.items(): print(k, v, v.exists())

train /content/garbage/train/images True
val /content/garbage/valid/images True
test /content/garbage/test/images True


In [ ]:
import re, shutil, yaml
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
images_rows=[]; boxes_rows=[]; issues=[]

def label_for(img, img_dir):
    # Standard YOLO dataset: split/images/foo.jpg -> split/labels/foo.txt
    try: relative=img.relative_to(img_dir)
    except ValueError: relative=Path(img.name)
    if img_dir.name == 'images':
        return img_dir.parent/'labels'/relative.with_suffix('.txt')
    return img_dir.parent/'labels'/relative.with_suffix('.txt')

for split, img_dir in SPLITS.items():
    image_paths=sorted(x for x in img_dir.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_EXTS)
    print(f'Scanning {split}: {len(image_paths)} images')
    for img in tqdm(image_paths, desc=split):
        label=label_for(img,img_dir)
        row={'split':split,'image':str(img),'label':str(label),'filename':img.name,'label_exists':label.exists(), 'image_ok':False, 'width':np.nan,'height':np.nan,'n_objects':0,'n_valid_objects':0,'invalid_rows':0}
        try:
            with Image.open(img) as im:
                row['width'],row['height']=im.size
                im.verify()
            row['image_ok']=True
        except Exception as e: issues.append({'split':split,'image':str(img),'issue':'unreadable_image','details':str(e)})
        if not label.exists():
            issues.append({'split':split,'image':str(img),'issue':'missing_label_file','details':''})
        else:
            try: lines=[x.strip() for x in label.read_text().splitlines() if x.strip()]
            except Exception as e:
                lines=[]; issues.append({'split':split,'image':str(img),'issue':'unreadable_label','details':str(e)})
            row['n_objects']=len(lines)
            for lineno,line in enumerate(lines,1):
                tokens=line.split()
                try:
                    if len(tokens)!=5: raise ValueError(f'expected 5 tokens, got {len(tokens)}')
                    cid_float=float(tokens[0]); cid=int(cid_float)
                    if cid_float != cid or cid not in NAMES: raise ValueError(f'unknown class ID {tokens[0]}')
                    x,y,w,h=map(float,tokens[1:])
                    if not all(np.isfinite([x,y,w,h])): raise ValueError('nonfinite coordinate')
                    if not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1): raise ValueError('out-of-range YOLO coordinate')
                    if x-w/2 < -1e-5 or x+w/2 > 1+1e-5 or y-h/2 < -1e-5 or y+h/2 > 1+1e-5: raise ValueError('bounding box extends outside image')
                    boxes_rows.append({'split':split,'image':str(img),'class_id':cid,'class_name':NAMES[cid], 'x':x,'y':y,'w':w,'h':h,'area':w*h,'width_px':w*row['width'],'height_px':h*row['height']})
                    row['n_valid_objects']+=1
                except Exception as e:
                    row['invalid_rows']+=1
                    issues.append({'split':split,'image':str(img),'issue':'invalid_annotation','details':f'line {lineno}: {e}'})
        images_rows.append(row)

images_df=pd.DataFrame(images_rows)
boxes_df=pd.DataFrame(boxes_rows, columns=['split','image','class_id','class_name','x','y','w','h','area','width_px','height_px'])
issues_df=pd.DataFrame(issues, columns=['split','image','issue','details'])
for name,df in [('images',images_df),('boxes',boxes_df),('issues',issues_df)]:
    df.to_csv(OUTPUT/f'{name}_audit.csv',index=False)
print('Scanned images:',len(images_df),'Valid boxes:',len(boxes_df),'Issues:',len(issues_df))
display(issues_df.groupby(['split','issue']).size().unstack(fill_value=0) if len(issues_df) else pd.DataFrame({'status':['No issues found']}))

Scanning train: 7324 images


train:   0%|          | 0/7324 [00:00<?, ?it/s]

Scanning val: 2098 images


val:   0%|          | 0/2098 [00:00<?, ?it/s]

Scanning test: 1042 images


test:   0%|          | 0/1042 [00:00<?, ?it/s]

Scanned images: 10464 Valid boxes: 74073 Issues: 0


,status
0,No issues found


In [ ]:
import re
from scipy.stats import ks_2samp

final = meta.copy()
final['split'] = final['orig_split']
final.loc[pool.index, 'split'] = pool['new_split']
final['prefix'] = final['stem'].map(lambda s: re.match(r'[A-Za-z]+', s).group(0).lower() if re.match(r'[A-Za-z]+', s) else '?')
S = ['train', 'valid', 'test']

# ---------- A) Integrity of the new split ----------
print('A) INTEGRITY')
print('Images per split:', final.split.value_counts().reindex(S).to_dict(), '| total:', len(final))
print('Duplicate image names:', final.image.duplicated().sum())
for a, b in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]:
    print(f'Shared source groups {a}-{b}:', len(set(final[final.split == a].group) & set(final[final.split == b].group)))

# ---------- B) Distribution match: every pair of splits ----------
def tv(a, b):
    pa, pb = final[final.split == a][C].sum(), final[final.split == b][C].sum()
    return round(0.5 * np.abs(pa / pa.sum() - pb / pb.sum()).sum(), 4)
def ks(a, b):
    return round(ks_2samp(final[final.split == a].n_obj, final[final.split == b].n_obj).statistic, 4)
print('\nB) DISTRIBUTION MATCH (lower = more similar)')
display(pd.DataFrame([{'pair': f'{a}-{b}', 'class TV': tv(a, b), 'density KS': ks(a, b)}
                      for a, b in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]]))

# ---------- C) Stability: is seed 0 just lucky? ----------
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
res = []
for seed in range(5):
    va, te = next(MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed).split(np.zeros(len(g)), Y))
    a = pd.Series('valid', index=g.index); a.iloc[te] = 'test'
    tmp = pool.assign(s=pool.group.map(a))
    pv, pt = tmp[tmp.s == 'valid'][C].sum(), tmp[tmp.s == 'test'][C].sum()
    res.append({'seed': seed, 'class TV': round(0.5 * np.abs(pv / pv.sum() - pt / pt.sum()).sum(), 4),
                'density KS': round(ks_2samp(tmp[tmp.s == 'valid'].n_obj, tmp[tmp.s == 'test'].n_obj).statistic, 4),
                'min test boxes': int(pt.min())})
print('\nC) STABILITY ACROSS SEEDS')
display(pd.DataFrame(res))

# ---------- D) Why the official split was skewed ----------
print('\nD) FILENAME SOURCE PREFIX x OFFICIAL SPLIT (images)')
display(pd.crosstab(final['prefix'], final['orig_split']).reindex(columns=S, fill_value=0))
print('FILENAME SOURCE PREFIX x NEW SPLIT (images)')
display(pd.crosstab(final['prefix'], final['split']).reindex(columns=S, fill_value=0))

A) INTEGRITY
Images per split: {'train': 7324, 'valid': 1571, 'test': 1569} | total: 10464
Duplicate image names: 0
Shared source groups train-valid: 0
Shared source groups train-test: 0
Shared source groups valid-test: 0

B) DISTRIBUTION MATCH (lower = more similar)


,pair,class TV,density KS
0,train-valid,0.0270,0.0126
1,train-test,0.0198,0.0123
2,valid-test,0.0382,0.0144



C) STABILITY ACROSS SEEDS


,seed,class TV,density KS,min test boxes
0,0,0.0382,0.0144,638
1,1,0.0562,0.0119,607
2,2,0.0682,0.0174,578
3,3,0.0434,0.0206,712
4,4,0.0563,0.0140,683



D) FILENAME SOURCE PREFIX x OFFICIAL SPLIT (images)


orig_split,train,valid,test
prefix,,,
biodegradable,1554,666,0
cardboard,1006,433,0
glass,1854,795,0
metal,874,204,170
paper,1223,0,523
plastic,813,0,349


FILENAME SOURCE PREFIX x NEW SPLIT (images)


split,train,valid,test
prefix,,,
biodegradable,1554,335,331
cardboard,1006,214,219
glass,1854,404,391
metal,874,186,188
paper,1223,258,265
plastic,813,174,175


In [ ]:
m = final[(final.prefix == 'metal') & final.orig_split.isin(['valid', 'test'])]
print('Last metal in valid :', m[m.orig_split == 'valid'].stem.max())
print('First metal in test :', m[m.orig_split == 'test'].stem.min())
print('Sequential cut confirmed:', m[m.orig_split == 'valid'].stem.max() < m[m.orig_split == 'test'].stem.min())

Last metal in valid : metal9_jpg.rf.e3905621bd2af8b2636cc66cc4102939
First metal in test : metal1000_jpg.rf.f972491130c59d3bff7e97b22e8e2980
Sequential cut confirmed: False


In [ ]:
m = final[(final.prefix == 'metal') & final.orig_split.isin(['valid', 'test'])].copy()
m['num'] = m.stem.str.extract(r'^[A-Za-z]+(\d+)')[0].astype(int)
m['hash'] = m.stem.str.split('.rf.').str[1]

for key in ['stem', 'num', 'hash']:
    seq = m.sort_values(key).orig_split.tolist()
    switches = sum(a != b for a, b in zip(seq, seq[1:]))
    print(f'Sorted by {key:5s} -> switches between valid/test: {switches}  (1 = clean sequential cut)')

print('\nvalid metal number range:', m[m.orig_split == 'valid'].num.min(), '-', m[m.orig_split == 'valid'].num.max())
print('test  metal number range:', m[m.orig_split == 'test'].num.min(), '-', m[m.orig_split == 'test'].num.max())

Sorted by stem  -> switches between valid/test: 3  (1 = clean sequential cut)
Sorted by num   -> switches between valid/test: 6  (1 = clean sequential cut)
Sorted by hash  -> switches between valid/test: 174  (1 = clean sequential cut)

valid metal number range: 6 - 998
test  metal number range: 4 - 1256


In [ ]:
import shutil, yaml
NEW_ROOT = Path('/content/garbage_resplit')
S = ['train', 'valid', 'test']

# ---------- Build ----------
assert adopt, 'Decision was KEEP — do not build.'
for s in ['valid', 'test']:
    for sub in ['images', 'labels']:
        d = NEW_ROOT / s / sub
        if d.exists(): shutil.rmtree(d)
        d.mkdir(parents=True)
tr = NEW_ROOT / 'train'
if tr.is_symlink(): tr.unlink()
if not tr.exists(): tr.symlink_to(DATASET_ROOT / 'train', target_is_directory=True)

for r in pool.itertuples():
    (NEW_ROOT / r.new_split / 'images' / r.image).symlink_to((DATASET_ROOT / r.orig_split / 'images' / r.image).resolve())
    lbl = DATASET_ROOT / r.orig_split / 'labels' / (r.stem + '.txt')
    if lbl.exists():
        (NEW_ROOT / r.new_split / 'labels' / (r.stem + '.txt')).symlink_to(lbl)

yaml.safe_dump({'path': str(NEW_ROOT), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
                'nc': len(names), 'names': names}, open(NEW_ROOT / 'data.yaml', 'w'), sort_keys=False)
pool[['image', 'group', 'orig_split', 'new_split']].to_csv(OUTPUT / 'resplit_assignment.csv', index=False)

# ---------- Verify ----------
rows, problems = [], []
for s in S:
    imgs = sorted((NEW_ROOT / s / 'images').iterdir())
    lbls = {p.stem for p in (NEW_ROOT / s / 'labels').iterdir()}
    broken = [p.name for p in imgs if not p.exists()]                       # dead symlinks
    no_lbl = [p.name for p in imgs if p.stem not in lbls]
    counts = {c: 0 for c in C}; n_box = 0; bad = 0
    for p in (NEW_ROOT / s / 'labels').iterdir():
        for line in p.read_text().splitlines():
            q = line.split()
            if not q: continue
            if len(q) != 5 or float(q[3]) <= 0 or float(q[4]) <= 0 or not all(0 <= float(v) <= 1 for v in q[1:]):
                bad += 1; continue
            counts[NAMES[int(q[0])]] += 1; n_box += 1
    rows.append({'split': s, 'images': len(imgs), 'labels': len(lbls), 'broken links': len(broken),
                 'images w/o label': len(no_lbl), 'invalid lines': bad, 'boxes': n_box, **counts})
chk = pd.DataFrame(rows).set_index('split')
display(chk)

# ---------- Compare with expected (from split cell) ----------
exp = final.groupby('split')[C].sum().reindex(S)
match = (chk[C] == exp).all().all()
print('Images match 7324/1571/1569:', chk['images'].tolist() == [7324, 1571, 1569])
print('Boxes per class match expected:', match)
print('No broken links / missing labels / invalid lines:',
      chk[['broken links', 'images w/o label', 'invalid lines']].sum().sum() == 0)
print('Total boxes:', chk['boxes'].sum(), '(expected 74073)')
print('yaml:', yaml.safe_load(open(NEW_ROOT / 'data.yaml')))

,images,labels,broken links,images w/o label,invalid lines,boxes,BIODEGRADABLE,CARDBOARD,GLASS,METAL,PAPER,PLASTIC
split,,,,,,,,,,,,
train,7324,7324,0,0,0,51601,31714,3385,5429,3948,2979,4146
valid,1571,1571,0,0,0,11561,6991,673,1335,1078,656,828
test,1569,1569,0,0,0,10911,6690,638,1045,815,752,971


Images match 7324/1571/1569: True
Boxes per class match expected: True
No broken links / missing labels / invalid lines: True
Total boxes: 74073 (expected 74073)
yaml: {'path': '/content/garbage_resplit', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 6, 'names': ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']}


In [ ]:
S = ['train', 'valid', 'test']
img_cls = {s: {c: 0 for c in C} for s in S}
box_cls = {s: {c: 0 for c in C} for s in S}
for s in S:
    for p in (NEW_ROOT / s / 'labels').iterdir():
        cls = [NAMES[int(l.split()[0])] for l in p.read_text().splitlines() if l.strip()]
        for c in cls: box_cls[s][c] += 1
        for c in set(cls): img_cls[s][c] += 1

def table(d):
    t = pd.DataFrame(d)[S]
    t['total'] = t.sum(axis=1)
    pct = t[S].div(t['total'], axis=0).mul(100).round(1).add_suffix(' %')
    return pd.concat([t, pct], axis=1)

print('IMAGES per class (an image counts once in each class it contains):')
display(table(img_cls))
print('BOXES per class:')
display(table(box_cls))

IMAGES per class (an image counts once in each class it contains):


,train,valid,test,total,train %,valid %,test %
BIODEGRADABLE,1603,342,342,2287,70.1,15.0,15.0
CARDBOARD,1053,224,225,1502,70.1,14.9,15.0
GLASS,1879,408,397,2684,70.0,15.2,14.8
METAL,1328,285,285,1898,70.0,15.0,15.0
PAPER,1238,265,265,1768,70.0,15.0,15.0
PLASTIC,1019,222,223,1464,69.6,15.2,15.2


BOXES per class:


,train,valid,test,total,train %,valid %,test %
BIODEGRADABLE,31714,6991,6690,45395,69.9,15.4,14.7
CARDBOARD,3385,673,638,4696,72.1,14.3,13.6
GLASS,5429,1335,1045,7809,69.5,17.1,13.4
METAL,3948,1078,815,5841,67.6,18.5,14.0
PAPER,2979,656,752,4387,67.9,15.0,17.1
PLASTIC,4146,828,971,5945,69.7,13.9,16.3


In [ ]:
!pip -q install imagehash
import hashlib, imagehash
from collections import Counter

recs, miss, dup_lines, dup_iou = [], [], [], []
for s in S:
    imgs = {p.stem: p for p in (NEW_ROOT / s / 'images').iterdir()}
    lbls = {p.stem: p for p in (NEW_ROOT / s / 'labels').iterdir()}
    for stem in imgs.keys() - lbls.keys(): miss.append((s, stem, 'image without label'))
    for stem in lbls.keys() - imgs.keys(): miss.append((s, stem, 'label without image'))
    for stem, p in tqdm(imgs.items(), desc=s):
        try:
            with Image.open(p) as im: im.verify()
            with Image.open(p) as im: ph = imagehash.phash(im.convert('RGB'))
        except Exception as e:
            miss.append((s, stem, f'unreadable image: {e}')); continue
        recs.append({'split': s, 'stem': stem, 'group': stem.split('.rf.')[0],
                     'md5': hashlib.md5(p.read_bytes()).hexdigest(), 'phash': ph})
        if stem not in lbls: continue
        lines = [l.strip() for l in lbls[stem].read_text().splitlines() if l.strip()]
        if not lines: miss.append((s, stem, 'empty label file'))
        ok = []
        for l in lines:
            q = l.split()
            try:
                v = list(map(float, q))
                if len(q) != 5 or not np.isfinite(v).all(): raise ValueError
                ok.append(v)
            except ValueError:
                miss.append((s, stem, f'malformed/NaN line: {l}'))
        for l, n in Counter(lines).items():
            if n > 1: dup_lines.append((s, stem, l, n))
        if len(ok) > 1:
            a = np.array(ok); c = a[:, 0]
            x1, y1, x2, y2 = a[:, 1] - a[:, 3]/2, a[:, 2] - a[:, 4]/2, a[:, 1] + a[:, 3]/2, a[:, 2] + a[:, 4]/2
            inter = (np.clip(np.minimum(x2[:, None], x2) - np.maximum(x1[:, None], x1), 0, None) *
                     np.clip(np.minimum(y2[:, None], y2) - np.maximum(y1[:, None], y1), 0, None))
            area = a[:, 3] * a[:, 4]
            iou = inter / (area[:, None] + area - inter)
            n_hi = int(((iou >= 0.95) & (c[:, None] == c) & np.triu(np.ones_like(iou, bool), 1)).sum())
            if n_hi: dup_iou.append((s, stem, n_hi))

R = pd.DataFrame(recs)

# ---------- 1) MISSING ----------
print('1) MISSING / CORRUPT')
miss_df = pd.DataFrame(miss, columns=['split', 'stem', 'problem'])
print('Total problems:', len(miss_df))
if len(miss_df): display(miss_df.groupby(['split', 'problem']).size().rename('count').to_frame())

# ---------- 2) DUPLICATES ----------
print('\n2) DUPLICATES')
print('Duplicate file names across dataset:', R.stem.duplicated().sum())
md5_dup = R[R.md5.duplicated(keep=False)]
print('Exact-duplicate image groups (MD5):', md5_dup.md5.nunique(), '| images involved:', len(md5_dup))
print('  ...of which span more than one split:', (md5_dup.groupby('md5').split.nunique() > 1).sum())
print('Label files with identical repeated lines:', len(dup_lines), '| extra lines:', sum(n - 1 for *_, n in dup_lines))
print('Label files with same-class boxes at IoU >= 0.95:', len(dup_iou), '| pairs:', sum(n for *_, n in dup_iou))

# ---------- 3) LEAKAGE ----------
print('\n3) LEAKAGE')
for a_, b_ in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]:
    print(f'Shared source groups {a_}-{b_}:', len(set(R[R.split == a_].group) & set(R[R.split == b_].group)))

H = np.array([np.packbits(h.hash.flatten()) for h in R.phash])
pop = np.array([bin(i).count('1') for i in range(256)], np.uint8)
spl, grp = R.split.values, R.group.values
pairs = []
for i0 in tqdm(range(0, len(H), 512), desc='near-dup'):
    d = pop[H[i0:i0+512, None, :] ^ H[None, :, :]].sum(-1)
    ii, jj = np.where(d <= 8); ii += i0
    m = jj > ii
    pairs += [(i, j, int(d[i - i0, j])) for i, j in zip(ii[m], jj[m])]
P = pd.DataFrame(pairs, columns=['i', 'j', 'dist'])
P['cross_split'] = spl[P.i] != spl[P.j]
P['pair'] = [' - '.join(sorted((spl[i], spl[j]))) for i, j in zip(P.i, P.j)]
summary = pd.DataFrame({f'dist <= {t}': P[P.dist <= t].groupby('pair').size() for t in [0, 4, 8]}).fillna(0).astype(int)
print('Near-duplicate image pairs by split pair (perceptual hash, Hamming distance out of 64):')
display(summary)
P.assign(img_i=R.stem.values[P.i], img_j=R.stem.values[P.j],
         split_i=spl[P.i], split_j=spl[P.j]).to_csv(OUTPUT / 'near_duplicate_pairs.csv', index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 6.9 MB/s eta 0:00:00


train:   0%|          | 0/7324 [00:00<?, ?it/s]

valid:   0%|          | 0/1571 [00:00<?, ?it/s]

test:   0%|          | 0/1569 [00:00<?, ?it/s]

1) MISSING / CORRUPT
Total problems: 0

2) DUPLICATES
Duplicate file names across dataset: 0
Exact-duplicate image groups (MD5): 0 | images involved: 0
  ...of which span more than one split: 0
Label files with identical repeated lines: 0 | extra lines: 0
Label files with same-class boxes at IoU >= 0.95: 20 | pairs: 20

3) LEAKAGE
Shared source groups train-valid: 0
Shared source groups train-test: 0
Shared source groups valid-test: 0


near-dup:   0%|          | 0/21 [00:00<?, ?it/s]

Near-duplicate image pairs by split pair (perceptual hash, Hamming distance out of 64):


,dist <= 0,dist <= 4,dist <= 8
pair,,,
test - test,3,7,17
test - train,3,5,49
test - valid,0,8,30
train - train,2,21,170
train - valid,3,11,66
valid - valid,2,2,18


In [ ]:
SAVE = OUTPUT / 'preprocessing'
SAVE.mkdir(parents=True, exist_ok=True)
saved = []

def save_df(name, df_fn):
    if all(v in globals() for v in name.split('+')):
        df = df_fn()
        df.to_csv(SAVE / f"{name.split('+')[0]}.csv", index=df.index.name is not None or not isinstance(df.index, pd.RangeIndex))
        saved.append(f"{name.split('+')[0]}.csv ({len(df)} rows)")

# Split
save_df('final',       lambda: final[['image', 'group', 'orig_split', 'split', 'n_obj'] + C])
save_df('pool',        lambda: pool[['image', 'group', 'orig_split', 'new_split']])
save_df('img_cls',     lambda: pd.DataFrame(img_cls)[S])
save_df('box_cls',     lambda: pd.DataFrame(box_cls)[S])
# Integrity / missing / duplicates / leakage
save_df('miss_df',     lambda: miss_df)
save_df('md5_dup',     lambda: md5_dup.drop(columns='phash'))
save_df('dup_lines',   lambda: pd.DataFrame(dup_lines, columns=['split', 'stem', 'line', 'count']))
save_df('dup_iou',     lambda: pd.DataFrame(dup_iou, columns=['split', 'stem', 'pairs_iou_ge_0.95']))
save_df('summary',     lambda: summary)
save_df('P+R',         lambda: P.assign(img_i=R.stem.values[P.i], img_j=R.stem.values[P.j],
                                        split_i=spl[P.i], split_j=spl[P.j]).drop(columns=['i', 'j']))

# Cleaned labels for all splits (so the split can be rebuilt after a runtime reset)
for s in S:
    shutil.make_archive(str(SAVE / f'labels_{s}'), 'zip', NEW_ROOT / s / 'labels')
    saved.append(f'labels_{s}.zip')
shutil.copy(NEW_ROOT / 'data.yaml', SAVE / 'data_resplit.yaml'); saved.append('data_resplit.yaml')

print('Saved to', SAVE)
for f in saved: print('  ', f)

Saved to /content/drive/MyDrive/garbage_outputs/preprocessing
   final.csv (10464 rows)
   pool.csv (3140 rows)
   img_cls.csv (6 rows)
   box_cls.csv (6 rows)
   miss_df.csv (0 rows)
   md5_dup.csv (0 rows)
   dup_lines.csv (0 rows)
   dup_iou.csv (20 rows)
   summary.csv (6 rows)
   P.csv (350 rows)
   labels_train.zip
   labels_valid.zip
   labels_test.zip
   data_resplit.yaml
